# 3 · Marriage


Presenter: Tongtong Tang

Original data file `marriage_data.csv` is now read from `Marriage_Data.csv` with identical values.

Data: `Marriage_Data.csv` (this member's columns of the group dataset).

In [1]:
from pathlib import Path
import os
import pandas as pd

# Locate this member's folder (the folder that contains Marriage_Data.csv).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Marriage_Data.csv').is_file())
COMBINED = pd.read_csv(ROOT / 'Marriage_Data.csv', float_precision='round_trip')

# Each section reads its own columns from the one combined dataset, under the column names its code expects.
SECTIONS = {
    'healthcare': dict(years=(2010, 2025), integer=['year', 'resident_live_births_number'], columns={
        'Year': 'year', 'TFR': 'tfr_per_female',
        'resident_live_births_number': 'resident_live_births_number',
        'cpi_health_2024_100': 'cpi_health_2024_100',
        'cpi_medicines_health_products_2024_100': 'cpi_medicines_health_products_2024_100',
        'cpi_outpatient_care_2024_100': 'cpi_outpatient_care_2024_100',
        'cpi_inpatient_care_2024_100': 'cpi_inpatient_care_2024_100',
        'cpi_other_health_services_2024_100': 'cpi_other_health_services_2024_100',
        'cpi_health_insurance_2024_100': 'cpi_health_insurance_2024_100',
        'cpi_polyclinic_fees_2024_100': 'cpi_polyclinic_fees_2024_100',
        'cpi_gp_fees_2024_100': 'cpi_gp_fees_2024_100',
        'cpi_specialist_outpatient_fees_2024_100': 'cpi_specialist_outpatient_fees_2024_100',
        'gov_health_development_sgd_million': 'gov_health_development_sgd_million',
        'gov_health_operating_sgd_million': 'gov_health_operating_sgd_million',
        'doctors_per_10000_total_population': 'doctors_per_10000_total_population',
        'nurses_per_10000_total_population': 'nurses_per_10000_total_population',
        'acute_hospital_beds_total_number': 'acute_hospital_beds_total_number',
        'acute_hospital_beds_public_number': 'acute_hospital_beds_public_number',
        'acute_hospital_beds_nonprofit_number': 'acute_hospital_beds_nonprofit_number',
        'acute_hospital_beds_private_number': 'acute_hospital_beds_private_number',
        'acute_hospitals_public_number': 'acute_hospitals_public_number',
        'acute_hospitals_private_number': 'acute_hospitals_private_number',
        'public_polyclinics_number': 'public_polyclinics_number',
        'young_children_health_inflation_2019base_pct': 'young_children_health_inflation_2019base_pct',
        'general_households_health_inflation_2019base_pct': 'general_households_health_inflation_2019base_pct',
        'Gov_Health_per_Capita': 'gov_health_per_capita_sgd',
        'Gov_Health_Total': 'gov_health_expenditure_sgd_million',
        'Gov_Health_%_GDP': 'gov_health_expenditure_gdp_pct',
        'HES_Health_Monthly': 'hes_all_households_health_monthly_sgd',
        'HES_Health_Share_%': 'hes_all_households_health_share_pct'},
        order=['year',
               'tfr_per_female',
               'resident_live_births_number',
               'cpi_health_2024_100',
               'cpi_medicines_health_products_2024_100',
               'cpi_outpatient_care_2024_100',
               'cpi_inpatient_care_2024_100',
               'cpi_other_health_services_2024_100',
               'cpi_health_insurance_2024_100',
               'cpi_polyclinic_fees_2024_100',
               'cpi_gp_fees_2024_100',
               'cpi_specialist_outpatient_fees_2024_100',
               'gov_health_per_capita_sgd',
               'gov_health_expenditure_sgd_million',
               'gov_health_expenditure_gdp_pct',
               'gov_health_development_sgd_million',
               'gov_health_operating_sgd_million',
               'doctors_per_10000_total_population',
               'nurses_per_10000_total_population',
               'hes_all_households_health_monthly_sgd',
               'hes_all_households_health_share_pct',
               'acute_hospital_beds_total_number',
               'acute_hospital_beds_public_number',
               'acute_hospital_beds_nonprofit_number',
               'acute_hospital_beds_private_number',
               'acute_hospitals_public_number',
               'acute_hospitals_private_number',
               'public_polyclinics_number',
               'young_children_health_inflation_2019base_pct',
               'general_households_health_inflation_2019base_pct']),
    'housing_hdb_quarterly': dict(years=(2000, 2025), integer=['Year'], columns={
        'Year': 'Year', 'HDB_Resale_Index_Q4': '4', 'HDB_Resale_Index_Q3': '3',
        'HDB_Resale_Index_Q2': '2', 'HDB_Resale_Index_Q1': '1'}),
    'housing_fertility': dict(years=(2000, 2025), integer=['Year'], columns={
        'Year': 'Year', 'TFR': 'Total_Fertility_Rate',
        **{f'ASFR_{a}_{a + 4}': f'{a} - {a + 4} Years' for a in range(15, 50, 5)}}),
    'housing_private': dict(years=(2000, 2025), integer=['Year'], columns={
        'Year': 'Year', 'Private_Landed_Index_URA': 'Private_Landed_Index',
        'Private_Non_Landed_Index_URA': 'Private_Non_Landed_Index'}),
    'marriage': dict(years=(2000, 2025), integer=['Year'], columns={c: c for c in [
        'Year', 'TFR', 'F_Marriage_Rate_20_24', 'F_Marriage_Rate_25_29', 'F_Marriage_Rate_30_34',
        'F_Marriage_Rate_35_39', 'F_Marriage_Rate_40_44', 'Median_Age_First_Birth', 'Brides_25_29', 'Brides_30_34']}),
    'income': dict(years=(2010, 2025), integer=['Year'], columns={c: c for c in ['Year', 'TFR', 'PM_Avg', 'PM_Median', 'CPI_All_Items_2024_100', 'CPI_All_Items_2010_100', 'PM_Avg_Real_2010', 'PM_Median_Real_2010', 'Real_Growth_%', 'Real_Growth_D1', 'Real_Growth_D2', 'Real_Growth_D3', 'Real_Growth_D4', 'Real_Growth_D5', 'Real_Growth_D6', 'Real_Growth_D7', 'Real_Growth_D8', 'Real_Growth_D9', 'Gini_Before_Transfers', 'Gini_After_Transfers', 'Real_Growth_D1_M810361', 'Real_Growth_D10_M810361']}),
    'education': dict(years=(2000, 2025), integer=['Year'],
                      columns={c: c for c in ['Year', 'TFR', 'ASFR_15_19', 'ASFR_20_24', 'ASFR_25_29', 'ASFR_30_34', 'ASFR_35_39', 'ASFR_40_44', 'Gini_Before_Transfers', 'Gini_After_Transfers', 'Income_Decile_1', 'Income_Decile_2', 'Income_Decile_3', 'Income_Decile_4', 'Income_Decile_5', 'Income_Decile_6', 'Income_Decile_7', 'Income_Decile_8', 'Income_Decile_9', 'F_Marriage_Rate_20_24', 'F_Marriage_Rate_25_29', 'F_Marriage_Rate_30_34', 'F_Marriage_Rate_35_39', 'F_Marriage_Rate_40_44', 'Median_Age_First_Birth', 'Brides_25_29', 'Brides_30_34', 'PM_Avg', 'PM_Median', 'Real_Growth_%', 'CPI_All_Items_2024_100', 'CPI_All_Items_2010_100', 'PM_Avg_Real_2010', 'PM_Median_Real_2010', 'Real_Growth_D1', 'Real_Growth_D2', 'Real_Growth_D3', 'Real_Growth_D4', 'Real_Growth_D5', 'Real_Growth_D6', 'Real_Growth_D7', 'Real_Growth_D8', 'Real_Growth_D9', 'Real_Growth_D1_M810361', 'Real_Growth_D10_M810361', 'HH_Size_Employed', 'HDB_Resale_Price_Index', 'Private_Landed_Index', 'Private_Non_Landed_Index', 'HDB_YoY_%', 'Landed_YoY_%', 'NonLanded_YoY_%', 'Edu_CPI', 'Edu_CPI_General', 'Edu_CPI_Tuition', 'Edu_CPI_Textbooks', 'Edu_Public_Total', 'Edu_Primary_per_Student', 'Edu_Secondary_per_Student', 'Edu_Op_Primary', 'Edu_Op_Secondary_JC', 'Edu_Op_Tertiary', 'Edu_Op_Other', 'Edu_Op_ITE', 'Edu_Development', 'Univ_Women_25_34_%', 'Univ_Women_35_44_%', 'Childcare_Places', 'Preschool_Age6_%', 'Pupils_per_Teacher', 'Health_CPI', 'Health_CPI_Medicines', 'Health_CPI_Outpatient', 'Health_CPI_Inpatient', 'Health_CPI_Other', 'Health_CPI_Insurance', 'HES_Health_Monthly', 'HES_Health_Share_%', 'Gov_Health_Total', 'Gov_Health_per_Capita', 'Gov_Health_%_GDP']}),
}


def load_section(name):
    spec = SECTIONS[name]
    y0, y1 = spec['years']
    frame = COMBINED.loc[COMBINED.Year.between(y0, y1), list(spec['columns'])]
    frame = frame.rename(columns=spec['columns']).reset_index(drop=True)
    if spec.get('order'):
        frame = frame[spec['order']]
    for column in spec['integer']:
        frame[column] = frame[column].astype('int64')
    return frame


print(f'Marriage_Data.csv: {COMBINED.shape[0]} years × {COMBINED.shape[1]} columns')

Marriage_Data.csv: 26 years × 10 columns


# AN6100 Final Project — Marriage Factor and TFR

This notebook studies how **marriage timing and family-formation patterns are associated with Singapore's Total Fertility Rate (TFR)** from 2010 to 2025.

It contains five interactive Marriage charts plus a presentation-focused analysis of the shift from age **25–29 toward 30–34**.

### Controls
- `YEAR_START`, `YEAR_END`
- `CORR_MODE`: `"Levels"` or `"Yearly change"`
- `LINK_SERIES`
- `LINK_MODE`
- `LAG`

> Correlation is used to describe association, not direct causation.


In [2]:

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import pearsonr

# -------------------- v7-style settings --------------------
YEAR_START = 2010
YEAR_END = 2025

CORR_MODE = "Levels"        # "Levels" or "Yearly change"
LINK_SERIES = "F_Marriage_Rate_20_24"
LINK_MODE = "Levels"        # "Levels" or "Yearly change"
LAG = 0                     # 0, 1, 2, or 3 years

BLUE = "#2a78d6"
ORANGE = "#eb6834"
VIOLET = "#4a3aa7"
RED = "#e34948"
DIVERGING = [[0, BLUE], [0.5, "#efeeea"], [1, RED]]

RATES = {
    "F_Marriage_Rate_20_24": "20-24",
    "F_Marriage_Rate_25_29": "25-29",
    "F_Marriage_Rate_30_34": "30-34",
    "F_Marriage_Rate_35_39": "35-39",
    "F_Marriage_Rate_40_44": "40-44",
}

SHORT = {
    "F_Marriage_Rate_20_24": "Marriage rate 20-24",
    "F_Marriage_Rate_25_29": "Marriage rate 25-29",
    "F_Marriage_Rate_30_34": "Marriage rate 30-34",
    "F_Marriage_Rate_35_39": "Marriage rate 35-39",
    "F_Marriage_Rate_40_44": "Marriage rate 40-44",
    "Median_Age_First_Birth": "Age at 1st birth",
    "Brides_25_29": "Brides 25-29",
    "Brides_30_34": "Brides 30-34",
    "TFR": "TFR",
}

def find_data():
    candidates = [
        Path("marriage_data.csv"),
        Path.cwd() / "marriage_data.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Put marriage_data.csv in the same folder as this notebook.")

raw = load_section('marriage')   # read from Marriage_Data.csv
df = raw[raw["Year"].between(YEAR_START, YEAR_END)].copy().sort_values("Year")
s = df.set_index("Year")

def corr_test(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    n = len(x)
    if n < 4 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan, n
    r, p = pearsonr(x, y)
    return float(r), float(p), n

def p_text(p):
    if np.isnan(p): return "–"
    if p < 0.001: return "<0.001"
    if p < 0.01: return f"{p:.3f}"
    return f"{p:.2f}"

def transform(frame, cols, mode):
    d = frame[cols].dropna()
    if mode == "Yearly change":
        full = d.reindex(range(int(d.index.min()), int(d.index.max()) + 1))
        return full.diff().dropna()
    return d

def corr_matrix(frame, cols, mode):
    m = transform(frame, cols, mode)
    c = pd.DataFrame(np.nan, index=cols, columns=cols)
    p = c.copy()
    n = c.copy()
    for a in cols:
        for b in cols:
            tmp = m[[a,b]].dropna()
            if a == b:
                rr, pp, nn = 1.0, 0.0, len(tmp)
            else:
                rr, pp, nn = corr_test(tmp[a], tmp[b])
            c.loc[a,b], p.loc[a,b], n.loc[a,b] = rr, pp, nn
    return c, p, n

def pair(data, x_col, mode="Levels", lag=0):
    ss = data.set_index("Year")
    xs = ss[x_col].dropna()
    ys = ss["TFR"].dropna()
    if lag:
        xs = xs.copy()
        xs.index = xs.index + lag
    d = pd.concat([xs.rename("x"), ys.rename("y")], axis=1).dropna()
    return transform(d, ["x","y"], mode)

peak = s[list(RATES)].dropna(how="all").idxmax(axis=1).map(RATES)
print(f"Selected years: {YEAR_START}–{YEAR_END} ({len(df)} observations)")


Selected years: 2010–2025 (16 observations)


## Figure 1 — Women's marriage rate by age

In [3]:

z = s[list(RATES)].dropna(how="all").T

fig = go.Figure(go.Heatmap(
    z=z.values,
    x=z.columns,
    y=list(RATES.values()),
    colorscale="Purples",
    xgap=1, ygap=2,
    text=np.round(z.values, 0),
    texttemplate="%{text:.0f}",
    colorbar=dict(title="per 1,000", thickness=10),
    hovertemplate="Age %{y}, %{x}: %{z:.1f}<extra></extra>"
))
fig.update_layout(
    title="Women's marriage rate by age",
    template="plotly_white",
    height=380
)
fig.update_yaxes(autorange="reversed", title_text="Age group")
fig.update_xaxes(title_text="Year")
fig.show()


### Insight

In [4]:
y0, y1 = int(df["Year"].min()), int(df["Year"].max())
r20_0 = s.loc[y0, "F_Marriage_Rate_20_24"]
r20_1 = s.loc[y1, "F_Marriage_Rate_20_24"]
r30_0 = s.loc[y0, "F_Marriage_Rate_30_34"]
r30_1 = s.loc[y1, "F_Marriage_Rate_30_34"]

print(
    f"From {y0} to {y1}, the female marriage rate at age 20–24 falls from "
    f"{r20_0:.1f} to {r20_1:.1f} per 1,000 unmarried women, while the 30–34 rate "
    f"moves from {r30_0:.1f} to {r30_1:.1f}. The peak marriage-rate group shifts "
    f"from {peak.iloc[0]} to {peak.iloc[-1]}, indicating later marriage timing."
)


From 2010 to 2025, the female marriage rate at age 20–24 falls from 22.6 to 13.7 per 1,000 unmarried women, while the 30–34 rate moves from 81.0 to 94.9. The peak marriage-rate group shifts from 25-29 to 30-34, indicating later marriage timing.


## Figure 2 — Peak marriage age and first-birth timing

In [5]:

PEAK_COL = {"25-29": BLUE, "30-34": ORANGE}
d = df.dropna(subset=["Median_Age_First_Birth"]).set_index("Year")
counts = [
    d.loc[y, "Brides_25_29"] if peak.get(y) == "25-29" else d.loc[y, "Brides_30_34"]
    for y in d.index
]

fig = go.Figure()
for yr, group in peak.items():
    fig.add_vrect(
        x0=yr-0.5, x1=yr+0.5,
        fillcolor=PEAK_COL.get(group, "#ffffff"),
        opacity=0.13, layer="below", line_width=0
    )

fig.add_trace(go.Scatter(
    x=d.index,
    y=d["Median_Age_First_Birth"],
    name="Median age at first birth",
    mode="lines+markers+text",
    text=[f"{v:.1f}" for v in d["Median_Age_First_Birth"]],
    textposition="top center",
    line=dict(color=VIOLET, width=2.6),
    marker=dict(size=6),
    customdata=np.c_[peak.reindex(d.index).values, counts],
    hovertemplate=(
        "%{x}: first birth at %{y:.1f}<br>"
        "peak marriage age %{customdata[0]} "
        "(%{customdata[1]:,.0f} brides)<extra></extra>"
    )
))

# TFR on the right axis
t = df.dropna(subset=["TFR"])
fig.add_trace(go.Scatter(
    x=t["Year"], y=t["TFR"],
    name="TFR",
    yaxis="y2",
    mode="lines+markers",
    line=dict(color=BLUE, width=2.2, dash="dot"),
    marker=dict(size=5, symbol="diamond"),
    hovertemplate="TFR %{y:.2f}<extra></extra>"
))

for group, color in PEAK_COL.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        name=f"Peak marriage age {group}",
        marker=dict(size=12, symbol="square", color=color, opacity=0.35)
    ))

lo, hi = t["TFR"].min(), t["TFR"].max()
pad = max((hi-lo)*0.15, 0.03)

fig.update_layout(
    title="Peak marriage age and first-birth timing",
    template="plotly_white",
    height=420,
    yaxis=dict(title="Mother's age at first birth (years)"),
    yaxis2=dict(
        title="TFR, children per woman",
        overlaying="y", side="right", showgrid=False,
        range=[lo-pad, hi+pad]
    ),
    legend=dict(orientation="h", y=1.10, x=0)
)
fig.update_xaxes(title_text="Year")
fig.show()


### Insight

In [6]:
fb0 = d["Median_Age_First_Birth"].iloc[0]
fb1 = d["Median_Age_First_Birth"].iloc[-1]
tfr0 = t["TFR"].iloc[0]
tfr1 = t["TFR"].iloc[-1]

print(
    f"The peak marriage age shifts from {peak.iloc[0]} to {peak.iloc[-1]}. "
    f"At the same time, median age at first birth rises from {fb0:.1f} to {fb1:.1f} years, "
    f"while TFR moves from {tfr0:.2f} to {tfr1:.2f}. This supports an association between "
    "later family formation and lower fertility, but does not prove causation."
)


The peak marriage age shifts from 25-29 to 30-34. At the same time, median age at first birth rises from 30.1 to 32.2 years, while TFR moves from 1.15 to 0.87. This supports an association between later family formation and lower fertility, but does not prove causation.


## Figure 3 — Brides aged 25–29 vs 30–34

In [7]:

d3 = df.dropna(subset=["Brides_25_29", "Brides_30_34"])

fig = go.Figure()
fig.add_trace(go.Bar(
    x=d3["Year"], y=d3["Brides_25_29"],
    name="25-29", marker_color=BLUE,
    hovertemplate="%{x}, brides 25-29: %{y:,.0f}<extra></extra>"
))
fig.add_trace(go.Bar(
    x=d3["Year"], y=d3["Brides_30_34"],
    name="30-34", marker_color=ORANGE,
    hovertemplate="%{x}, brides 30-34: %{y:,.0f}<extra></extra>"
))
fig.update_layout(
    title="Brides aged 25-29 vs 30-34",
    template="plotly_white",
    height=380,
    barmode="group"
)
fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="Marriages (brides)")
fig.show()


### Insight

In [8]:
print(
    f"Actual brides aged 25–29 change from {d3['Brides_25_29'].iloc[0]:,.0f} to "
    f"{d3['Brides_25_29'].iloc[-1]:,.0f}, while brides aged 30–34 change from "
    f"{d3['Brides_30_34'].iloc[0]:,.0f} to {d3['Brides_30_34'].iloc[-1]:,.0f}. "
    "These counts reinforce the ageing pattern, although marriage rates are still the better "
    "measure for comparing marriage propensity because counts depend on age-group population size."
)


Actual brides aged 25–29 change from 10,748 to 9,601, while brides aged 30–34 change from 5,185 to 7,444. These counts reinforce the ageing pattern, although marriage rates are still the better measure for comparing marriage propensity because counts depend on age-group population size.


## Figure 4 — Key correlations: marriage timing and TFR

In [9]:

KEY = [
    "F_Marriage_Rate_20_24",
    "F_Marriage_Rate_30_34",
    "Median_Age_First_Birth",
    "TFR"
]
c, p, n = corr_matrix(s, KEY, CORR_MODE)
names = [SHORT[k] for k in KEY]

text = []
hover = []
for i in range(len(KEY)):
    tr, hr = [], []
    for j in range(len(KEY)):
        sig = ""
        if i != j and not np.isnan(p.iloc[i,j]):
            sig = "**" if p.iloc[i,j] < 0.01 else "*" if p.iloc[i,j] < 0.05 else ""
        tr.append(f"{c.iloc[i,j]:.2f}{sig}" if not np.isnan(c.iloc[i,j]) else "–")
        hr.append(f"p = {p_text(p.iloc[i,j])}, {int(n.iloc[i,j])} years")
    text.append(tr)
    hover.append(hr)

fig = go.Figure(go.Heatmap(
    z=c.values, x=names, y=names,
    colorscale=DIVERGING, zmin=-1, zmax=1,
    xgap=3, ygap=3,
    text=text, texttemplate="%{text}",
    customdata=hover,
    hovertemplate="%{y} vs %{x}: r = %{z:.2f}<br>%{customdata}<extra></extra>"
))
fig.update_layout(
    title=f"Key correlations: marriage timing and TFR — {CORR_MODE}",
    template="plotly_white",
    height=430
)
fig.update_yaxes(autorange="reversed")
fig.show()


### Insight

In [10]:
r20 = c.loc["F_Marriage_Rate_20_24", "TFR"]
r30 = c.loc["F_Marriage_Rate_30_34", "TFR"]
rfb = c.loc["Median_Age_First_Birth", "TFR"]

print(
    f"Using {CORR_MODE.lower()}, the 20–24 marriage rate vs TFR is r={r20:+.2f}, "
    f"the raw 30–34 marriage rate vs TFR is r={r30:+.2f}, and median age at first birth "
    f"vs TFR is r={rfb:+.2f}. The raw 30–34 rate alone does not capture the timing shift, "
    "which is why the relative movement from 25–29 toward 30–34 is analysed separately below."
)


Using levels, the 20–24 marriage rate vs TFR is r=+0.89, the raw 30–34 marriage rate vs TFR is r=-0.01, and median age at first birth vs TFR is r=-0.88. The raw 30–34 rate alone does not capture the timing shift, which is why the relative movement from 25–29 toward 30–34 is analysed separately below.


## Figure 5 — Marriage vs TFR

In [11]:

# Figure 5 now shows ALL female age-specific marriage-rate groups vs TFR.

AGE_RATE_SERIES = [
    "F_Marriage_Rate_20_24",
    "F_Marriage_Rate_25_29",
    "F_Marriage_Rate_30_34",
    "F_Marriage_Rate_35_39",
    "F_Marriage_Rate_40_44",
]

age_summary = []

for series in AGE_RATE_SERIES:
    d5 = pair(df, series, LINK_MODE, LAG)

    if len(d5) < 4:
        print(f"{SHORT[series]}: not enough overlapping years.")
        continue

    r5, p5, n5 = corr_test(d5["x"], d5["y"])

    # Regression line
    b, a = np.polyfit(d5["x"], d5["y"], 1)
    xs = np.linspace(d5["x"].min(), d5["x"].max(), 50)

    fig = go.Figure()

    if LINK_MODE != "Levels":
        fig.add_hline(y=0, line=dict(color="#cccccc", width=1))
        fig.add_vline(x=0, line=dict(color="#cccccc", width=1))

    fig.add_trace(
        go.Scatter(
            x=xs,
            y=a + b * xs,
            mode="lines",
            line=dict(color="#777777", dash="dash", width=1.4),
            hoverinfo="skip",
            showlegend=False,
        )
    )

    years = d5.index.values
    ends = {years.min(), years.max()}

    fig.add_trace(
        go.Scatter(
            x=d5["x"],
            y=d5["y"],
            mode="markers+text",
            text=[str(y) if y in ends else "" for y in years],
            textposition="top center",
            customdata=years,
            marker=dict(
                size=13,
                color=years,
                colorscale="Blues",
                line=dict(color="white", width=1.5),
                showscale=True,
                colorbar=dict(title="Year"),
            ),
            hovertemplate=(
                "<b>%{customdata}</b><br>"
                + SHORT[series]
                + ": %{x:,.2f}<br>TFR: %{y:,.3f}<extra></extra>"
            ),
            showlegend=False,
        )
    )

    suffix = "" if LINK_MODE == "Levels" else " (change vs previous year)"

    fig.update_layout(
        title=f"{SHORT[series]} vs TFR",
        template="plotly_white",
        height=420,
        margin=dict(l=70, r=50, t=65, b=60),
    )
    fig.update_xaxes(title_text=SHORT[series] + suffix)
    fig.update_yaxes(
        title_text=("TFR" + (f" {LAG} yr later" if LAG else "")) + suffix
    )

    fig.show()

    significance = (
        "significant" if not np.isnan(p5) and p5 < 0.05
        else "not significant"
    )

    print(
        f"Insight — {SHORT[series]}: r={r5:+.2f}, "
        f"p={p_text(p5)}, n={n5} ({significance})."
    )

    age_summary.append({
        "Age group": SHORT[series].replace("Marriage rate ", ""),
        "r vs TFR": round(r5, 3),
        "p-value": p_text(p5),
        "n": n5,
        "Significant (p<0.05)": "Yes" if significance == "significant" else "No",
    })

# Summary table across all age groups
age_summary_df = pd.DataFrame(age_summary)
display(age_summary_df)


Insight — Marriage rate 20-24: r=+0.89, p=<0.001, n=16 (significant).


Insight — Marriage rate 25-29: r=+0.74, p=0.001, n=16 (significant).


Insight — Marriage rate 30-34: r=-0.01, p=0.96, n=16 (not significant).


Insight — Marriage rate 35-39: r=-0.51, p=0.04, n=16 (significant).


Insight — Marriage rate 40-44: r=-0.36, p=0.17, n=16 (not significant).


,Age group,r vs TFR,p-value,n,Significant (p<0.05)
0,20-24,0.886,<0.001,16,Yes
1,25-29,0.739,0.001,16,Yes
2,30-34,-0.013,0.96,16,No
3,35-39,-0.512,0.04,16,Yes
4,40-44,-0.359,0.17,16,No


### Insight

In [12]:

# Overall Figure 5 interpretation
if not age_summary_df.empty:
    strongest = age_summary_df.iloc[age_summary_df["r vs TFR"].abs().argmax()]
    print(
        "Overall insight: the relationship with TFR is not the same across age groups. "
        f"The strongest absolute correlation in the selected setting is for age {strongest['Age group']} "
        f"(r={strongest['r vs TFR']:+.2f}, p={strongest['p-value']}). "
        "This comparison helps distinguish which part of the marriage-age profile is most closely associated with TFR."
    )


Overall insight: the relationship with TFR is not the same across age groups. The strongest absolute correlation in the selected setting is for age 20-24 (r=+0.89, p=<0.001). This comparison helps distinguish which part of the marriage-age profile is most closely associated with TFR.


## Figure 6 — Relative shift toward age 30–34 vs TFR

To measure **later marriage timing** more directly, this figure uses a relative shift index:

\[
	ext{Later-marriage shift index}
=
rac{	ext{Female marriage rate age 30–34}}
{	ext{Female marriage rate age 25–29}}
\]

- **Index < 1:** the 25–29 marriage rate is still higher.
- **Index > 1:** the 30–34 marriage rate is higher.

This is more useful than the raw 30–34 marriage rate alone because it measures the **relative movement of the marriage-rate peak toward an older age group**.


In [13]:

shift = df[
    ["Year", "TFR", "F_Marriage_Rate_25_29", "F_Marriage_Rate_30_34"]
].dropna().copy()

shift["Later_Marriage_Shift_Index"] = (
    shift["F_Marriage_Rate_30_34"] /
    shift["F_Marriage_Rate_25_29"]
)

r_shift, p_shift, n_shift = corr_test(
    shift["Later_Marriage_Shift_Index"],
    shift["TFR"]
)

# Best-fit line
b, a = np.polyfit(
    shift["Later_Marriage_Shift_Index"],
    shift["TFR"],
    1
)

xs = np.linspace(
    shift["Later_Marriage_Shift_Index"].min(),
    shift["Later_Marriage_Shift_Index"].max(),
    100
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=xs,
        y=a + b * xs,
        mode="lines",
        line=dict(
            color="#808080",
            dash="dash",
            width=2
        ),
        hoverinfo="skip",
        showlegend=False
    )
)

# Reference line: x = 1 means both age groups have the same marriage rate
fig.add_vline(
    x=1,
    line_dash="dot",
    line_color="#8a8984",
    line_width=1.2
)

fig.add_trace(
    go.Scatter(
        x=shift["Later_Marriage_Shift_Index"],
        y=shift["TFR"],
        mode="markers+text",
        text=shift["Year"].astype(int).astype(str),
        textposition="top center",
        textfont=dict(size=9),
        customdata=shift["Year"].astype(int),
        marker=dict(
            size=12,
            color=shift["Year"],
            colorscale="Oranges",
            cmin=shift["Year"].min(),
            cmax=shift["Year"].max(),
            line=dict(color="white", width=1.2),
            colorbar=dict(title="Year")
        ),
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Later-marriage shift index: %{x:.3f}<br>"
            "TFR: %{y:.2f}<extra></extra>"
        ),
        showlegend=False
    )
)

fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.02,
    y=0.02,
    text=f"<b>r = {r_shift:.2f}, p {p_text(p_shift)}, n = {n_shift}</b>",
    showarrow=False,
    font=dict(size=14),
    align="left"
)

fig.update_layout(
    title="Relative shift toward age 30–34 vs TFR",
    template="plotly_white",
    height=520,
    margin=dict(l=70, r=90, t=70, b=70)
)

fig.update_xaxes(
    title_text="Later-marriage shift index = rate(30–34) / rate(25–29)",
    showgrid=True
)

fig.update_yaxes(
    title_text="TFR",
    showgrid=True
)

fig.show()


### Insight

In [14]:

first = shift.iloc[0]
last = shift.iloc[-1]

significance = (
    "statistically significant"
    if p_shift < 0.05
    else "not statistically significant"
)

print(
    f"The later-marriage shift index increases from "
    f"{first['Later_Marriage_Shift_Index']:.2f} in {int(first['Year'])} to "
    f"{last['Later_Marriage_Shift_Index']:.2f} in {int(last['Year'])}. "
    f"Across {int(first['Year'])}–{int(last['Year'])}, the index is negatively correlated "
    f"with TFR (r={r_shift:+.2f}, p={p_text(p_shift)}, n={n_shift}), and the relationship is "
    f"{significance}. A higher index means marriage is relatively more concentrated among women "
    f"aged 30–34 than 25–29. This supports an association between later family formation and lower TFR, "
    "but it does not prove that later marriage directly causes lower fertility."
)


The later-marriage shift index increases from 0.82 in 2010 to 1.17 in 2025. Across 2010–2025, the index is negatively correlated with TFR (r=-0.75, p=<0.001, n=16), and the relationship is statistically significant. A higher index means marriage is relatively more concentrated among women aged 30–34 than 25–29. This supports an association between later family formation and lower TFR, but it does not prove that later marriage directly causes lower fertility.
